## Prompt:

TASK DESCRIPTION:
  * This is an IMAGE-LEVEL BINARY CLASSIFICATION task implemented using an object detection model.
  * The goal is to determine whether an image contains an animal or not.

DATASET STRUCTURE:
  * DATASET_ROOT contains three subdirectories: train, test, and val.
  * Each directory contains two subdirectories:
  * images/ → contains image files (.jpg, .jpeg, .png)
  * labels/ → contains YOLO format .txt files

GROUND-TRUTH LOGIC: 
  * An image is considered an animal if a corresponding .txt file exists and is not empty in the labels/ folder.
  * A non-empty file is a file whose size is larger than 0, and the size of an empty image is 0.

MODEL REQUIREMENTS:
  * Use ONLY a pretrained Ultralytics YOLO detection model (e.g., yolov8n.pt).
  * Call our RESTful API for yolo inference.
  * Assume YOLO detects animals using class ID animal at index 0.

YOLO INFERENCE APIs:

  * Sample CURL Request:
    ```
    curl -sS -X POST '${BASEURL}/v1/yolo/infer' \
    -H 'Authorization: Bearer ${FLEXSERV_TOKEN}' \
    -H 'Content-Type: application/json' \
    -d '{"model":"${FLEXSERV_MODEL_ID}","task":"detect","source":{"type":"upload","media_type":"image","content_base64":"data:image/jpeg;base64,/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBwcJCQgKDBQNDAsLDBkSEw8UHRofHh0aHBwgJC4nICIsIxwcKDcpLDAxNDQ0Hyc5PTgyPC4zNDL","filename":"NOR3__2019-07-19__11-40-00-1-_JPG.rf.b85ee30f99a803b09f8c5a7da7f9a508.jpg"},"params":{"conf":0.25,"iou":0.7,"imgsz":640,"max_det":300,"show_labels":true,"show_conf":true},"response":{"include":["predictions","timing"],"box_format":"xyxy","classification_topk":5,"return_original_shape":true}}' 
    ```

  * RESPONSE: 
    ```
    {
    "object": "yolo.inference",
    "task": "detect",
    "model": "/app/models/private/yolo--yolo26l/model.pt",
    "media_type": "image",
    "predictions": [
        {
        "frame_index": 0,
        "path": "image0.jpg",
        "original_shape": {
            "height": 640,
            "width": 640
        },
        "detections": [
            {
            "class_id": 61,
            "class_name": "toilet",
            "confidence": 0.480474591255188,
            "bbox": [
                0,
                29.76239013671875,
                637.2445068359375,
                629.2056274414062
            ],
            "bbox_format": "xyxy",
            "track_id": null
            }
        ]
        }
    ],
    "timing": {
        "inference_ms": 21.03
    },
    "annotated_media": null,
    "annotated_media_mime_type": null,
    "annotated_media_filename": null,
    "warnings": []
    }
    ```
For each image, one object in the 'predictions' array, any if anything detected, the 'detections' array will contain
a list of detected objects, and if nothing detected, there won't be 'detections' array. 
If any detected object is with class_id=0, an animal is detected. 


DETECTION LOGIC (IMPORTANT):

  * Run object detection on each image.
  * If the model produces AT LEAST ONE detection of an animal class with confidence >= 0.5 and IoU >= 0.7:
  *   → The image-level prediction is animal.

EVALUATION METRICS:

  * Iterate through the images in the test split.
  * Compare the image-level prediction with the ground truth (existence of label file).
  * Count: True Positives, True Negatives, False Positives, and False Negatives.

ACCURACY DEFINITION:

  * Overall accuracy = (True Positives + True Negatives) / Total Images

OUTPUT REQUIREMENTS:

  * Print for each image: filename, ground-truth status, and prediction.
  * At the end, print a summary report including total images, counts for each metric, and overall detection accuracy.

CODING REQUIREMENTS:

  * Store the main path in a global varaible DATASET_ROOT.
  * Set global variable for BASEURL and Bearer Auth Token (i.e. FLEXSERV_TOKEN).
  * Set global variable for BASE_YOLO_MODEL and FINE_TUNED_YOLO_MODEL, and also a MODEL_TO_USE for easy model switching.
  * Set global variable for confidence threashold and IoU threashold.
  * Make sure we disable SSL/TLS verification and also disable the related warning.
  * Make sure we pass image_name into the yolo inference request.
  * Make sure we pass the correct header for auth token and content-type in the final request.
  * Make sure we pass the request body correctly in the final request.
  * The `content_base64` field in the request should start with "data:image/jpeg;base64," and then appended with the base64 encoded image data.
  * Use pathlib or os for robust file path matching.
  * Read only .jpg files.
  * For inference of each image file, print the number of the image versus total number of images, the time spent for each inference request versus the total time spent for the entire inference step (in ms), the ground truth and detection result. 
  * Include clear comments explaining each step.
  * Output the accuracy in percentage format.
  * Don't use any mock or dummy functions. Make sure every line functions. 
  * It is okay to capture general Exception instead of every single type of Exceptions.

DEFENSIVE PROGRAMMING
  In case of any unexpected conditions, make sure the following: 
      1. Make sure we don't do SSL/TLS verification when sending request. 
      2. Make sure we avoid zero division

FACTS TO KNOW: 
  * BASEURL of FLEXSERV inference engine: https://vista.tacc.utexas.edu:60324
  * Bearer Auth token for FLEXSERV inference engine: 31b8148f20a4e8749dc232b48158a64b93ac7a988bd6aba5cc5de90c5654f984
  * FLEXSERV model ID format: FLEX:{PUB|PRI}:author/model[@revision], we only use private model pool, and omit the revision in model ID. 
  * DATASET_ROOT address: /home/jovyan/ai-tutorial-2026/datasets/AnimalEcology.v4i.yolov11
  * BASE_YOLO_MODEL for the request:  FLEX:PRI:yolo/yolo26n
  * FINE_TUNED_YOLO_MODEL for the request:  FLEX:PRI:yolo/yolo26n-fine-tuned

After the code, briefly explain how the program works in plain English.

## Sample code


In [2]:
import os
from pathlib import Path
import torch
from ultralytics import YOLO

# Define the dataset root directory
DATASET_ROOT = Path('/home/jovyan/work/vista/ai-tutorial-2026/datasets/AnimalEcology.v4i.yolov11')  # Update this path to your dataset root

# Load the pretrained YOLOv8 model
model = YOLO('/home/jovyan/work/vista/ai-tutorial-2026/models/yolov9t_ep200_bs32_lr0.005_baa22147.pt')

def load_images_and_labels(test_dir):
    """
    Load image paths and their corresponding label paths.
    """
    images = []
    labels = []
    
    # Iterate over all .jpg files in the images directory
    for img_path in test_dir.glob('images/*.jpg'):
        # Construct the corresponding label file path
        label_path = test_dir / 'labels' / f"{img_path.stem}.txt"
        
        images.append(img_path)
        labels.append(label_path)
    
    return images, labels

def predict_with_yolo(model, img_path):
    """
    Perform object detection on an image using the YOLO model.
    Returns True if an animal is detected with confidence >= 0.5, else False.
    """
    results = model.predict(source=str(img_path), conf=0.5)
    for result in results:
        boxes = result.boxes.cpu().numpy()
        for box in boxes:
            cls = int(box.cls[0])
            conf = box.conf[0]
            if cls == 0 and conf >= 0.5:  # Assuming class ID 0 is for animals
                return True
    return False

def evaluate_model(images, labels):
    """
    Evaluate the model's predictions against the ground truth.
    """
    true_positives = 0
    true_negatives = 0
    false_positives = 0
    false_negatives = 0
    
    for img_path, label_path in zip(images, labels):
        # Determine ground truth
        ground_truth = label_path.exists() and label_path.stat().st_size > 0
        
        # Get model prediction
        prediction = predict_with_yolo(model, img_path)
        
        # Compare prediction with ground truth
        if ground_truth and prediction:
            true_positives += 1
        elif not ground_truth and not prediction:
            true_negatives += 1
        elif not ground_truth and prediction:
            false_positives += 1
        elif ground_truth and not prediction:
            false_negatives += 1
        
        # Print result for each image
        print(f"Filename: {img_path.name}, Ground Truth: {'Animal' if ground_truth else 'No Animal'}, Prediction: {'Animal' if prediction else 'No Animal'}")
    
    return true_positives, true_negatives, false_positives, false_negatives

def main():
    # Load test images and labels
    test_dir = DATASET_ROOT / 'test'
    images, labels = load_images_and_labels(test_dir)
    
    # Evaluate the model
    tp, tn, fp, fn = evaluate_model(images, labels)
    
    # Calculate overall accuracy
    total_images = len(images)
    accuracy = (tp + tn) / total_images
    
    # Print summary report
    print("\nSummary Report:")
    print(f"Total Images: {total_images}")
    print(f"True Positives: {tp}")
    print(f"True Negatives: {tn}")
    print(f"False Positives: {fp}")
    print(f"False Negatives: {fn}")
    print(f"Overall Accuracy: {accuracy:.2f}")

if __name__ == "__main__":
    main()

In [ ]:
# Define the dataset root directory
DATASET_ROOT = Path('/home/jovyan/work/vista/ai-tutorial-2026/datasets/AnimalEcology.v4i.yolov11')  # Update this path to your dataset root

# Load the pretrained YOLOv8 model
model = YOLO('/home/jovyan/work/vista/ai-tutorial-2026/models/yolov9t_ep200_bs32_lr0.005_baa22147.pt')


## Paste your generated Code here 
